# Next step: step 11, the command line
**What it adds:** running SIMR from a shell or a Slurm script, with no Python:

```bash
python -m galactic_tidal_constraints.simr --models models.txt --output runs/ [--config config.json] [--log-level INFO]
```

| Flag | |
|---|---|
| `--models` | Required. One model per line: the first 14 comma-separated values (longer lines are fine, counted in one log message) |
| `--output` | Required: a folder or a `.json` catalog file, so every run is recorded |
| `--config` | Optional. Only what differs from the defaults. `run.return` is ignored with a warning |
| `--log-level` | `DEBUG`, `INFO` (default), `WARNING`, `ERROR` |

**Exit codes:** 0 the run ran (even with some failed simulations), 1 it couldn't run (a short message, no traceback),
2 every simulation failed.

Also new, for notebooks too: `simr.read_models(path)`, and **type checks** on the request: `"lnl": "0.1"` is now
`simulator[0].lnl must be a number: '0.1'` instead of a confusing comparison error.

**Until fitness exists, a run computes only what `run.save` asks for,** so a command-line run without `run.save`
records its models and configs but simulates nothing; each simulation's status is then `"empty"`.

## What changed
**New `src/galactic_tidal_constraints/simr/__main__.py`** (what `python -m galactic_tidal_constraints.simr` runs):
```python
def main(argv=None) -> int:
    args = _parser().parse_args(argv)
    logging.basicConfig(level=args.log_level, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    try:
        models = simr.read_models(args.models)
        request = _read_config(args.config)          # {} without --config; drops run.return with a warning
        result = simr.run(models, request, output=args.output)
    except (OSError, ValueError) as error:           # a bad input: a short message, no traceback
        logger.error("%s", error)
        return COULD_NOT_RUN                         # 1
    status = result.status["sim"]
    if status.size and (status == "failed").all():
        return ALL_FAILED                            # 2
    return RAN                                       # 0
```
**`simr.py`:** `read_models(path)`, and in `_simulator_config` a check of every value against `SimulatorConfig`'s
type hints:
```python
hints = typing.get_type_hints(SimulatorConfig)
for key, value in entry.items():
    if not _has_type(value, hints[key]):
        raise ValueError(f"simulator[{index}].{key} must be {_type_name(hints[key])}: {value!r}")
```
`_has_type` accepts an int for a float field (JSON writes `1.0` as `1`), never `true` / `false` for numbers, and
either option of a union like `start: str | float`.

**Tests:** new `tests/test_cli.py`; `tests/test_simr.py`, section "Step 11". **Docs:** `docs/design/simr.md`, The
command line. **Decisions:** `PLAN.md`, Configuration.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

from galactic_tidal_constraints.simr import simr

work = Path(tempfile.mkdtemp(prefix="simr_11_"))


def simr_cli(*args):
    """Run the command line as a shell would; print its log and exit code."""
    result = subprocess.run([sys.executable, "-m", "galactic_tidal_constraints.simr", *map(str, args)],
                            capture_output=True, text=True)
    print(result.stderr.strip())
    print("exit code:", result.returncode)
    return result.returncode


# Three lines of JSPAM's archive/data.txt, as they are: 34 values, then the start, the plot range and the name.
models_file = work / "models.txt"
models_file.write_text("""# three Galaxy Zoo: Mergers best fits
-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0 -6.904942981293307 24.945037352311584 SDSS_587722984435351614
1.08119,-1.66798,-2.06826,-0.18438,-0.39693,-0.02919,0.51654,0.36192,0.94043,0.43877,12.97415,135.11902,97.54035,305.01225,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.68139,-4.67999,1.02298,0.0,0.0,1.0,0.0,0.0,0.0 -9.639999999999883 2.8158787054319507 SDSS_587724234257137777
-0.49344,-1.43805,0.05985,0.07349,-1.02296,0.89921,1.09707,1.54176,1.27761,0.33429,48.78911,43.8006,50.06943,135.05795,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,5.08621,-0.9,0.43029,0.0,0.0,0.0,0.0,0.0,0.0 -5.0 3.6092302720033733 SDSS_587726033843585146
""")
print(simr.read_models(models_file))

## 1. A run with a config
Only what differs from the defaults: save the orbit, at two `lnl` values:

In [ ]:
config = work / "config.json"
config.write_text(json.dumps({"run": {"save": ["sim.orbit"]}, "simulator": [{"lnl": 0.1}, {"lnl": 0.001}]}, indent=2))
simr_cli("--models", models_file, "--config", config, "--output", work / "runs")

catalog = simr.load_catalog(work / "runs")
print("\nmodels:", list(catalog["models"]), "  status:", catalog["runs"][0]["status"])
print("files:", sorted(str(p.relative_to(work)) for p in (work / "runs").rglob("*.npy")))

## 2. No config, and a `.json` output
Without a config there's one default simulator entry, and nothing to save, so only the catalog is written (and, until
fitness exists, nothing is simulated: every status is `"empty"`):

In [ ]:
simr_cli("--models", models_file, "--output", work / "gen_01.json", "--log-level", "WARNING")
record = simr.load_catalog(work / "gen_01.json")["runs"][0]
print("recorded config:", {key: record["simulator"][0][key] for key in ("seed", "n1", "lnl", "start")}, "...")
print("status:", record["status"])

## 3. Warnings and exit codes

In [ ]:
with_return = work / "with_return.json"
with_return.write_text(json.dumps({"run": {"return": ["sim.orbit"], "save": ["sim.timing"]}}))
print("--- run.return in the config: a warning, and the run goes on")
simr_cli("--models", models_file, "--config", with_return, "--output", work / "runs_2")

failing = work / "failing.json"
failing.write_text(json.dumps({"run": {"save": ["sim.timing"]}, "simulator": {"start_pass": 5, "no_pass": "error"}}))
print("\n--- every simulation fails (no 5th pass, and no_pass is 'error'): exit 2")
simr_cli("--models", models_file, "--config", failing, "--output", work / "runs_3", "--log-level", "ERROR")

bad = work / "bad.json"
bad.write_text(json.dumps({"simulator": {"lnl": "0.1"}}))
print("\n--- a wrong type in the config: exit 1, a short message")
simr_cli("--models", models_file, "--config", bad, "--output", work / "runs_4")

print("\n--- an output with another extension: exit 1")
simr_cli("--models", models_file, "--output", work / "runs.txt")